# TPDS - Exp065: replay of the selected submission (final model layer)

Purpose: re-run the final model layer of the 9th-place submission (team seantangth) from the published checkpoint and check the md5 of the two selected files.  
Runtime: CPU - Kaggle CPU notebook, **Internet ON** (installs Python 3.13 and the pinned packages)  
Why: LightGBM / CatBoost / polars only; no GPU code  
Estimated time: 60–90 min on Kaggle CPU (about 30 min on a 10-core Apple M4)  
Inputs: competition data; dataset `tpds-9th-place-artifacts` (`checkpoint.tar`, `code.tar`, the two submitted CSVs)  
Outputs (in `/kaggle/working`): `submission.csv` (= A), the re-built B and base files, `logs_replay/`, `replay_summary.json`; the work tree itself stays in `/tmp` so that the 3.5 GB checkpoint is not saved as notebook output  
Prerequisites: None

What is re-run (`replay.sh`): the stage-1 pair ensemble (LightGBM + CatBoost, training worlds F and FW), the stage-2 head re-ranker and the D′ rank rule, the family classifier, the evidence-listing decoder (12-seed detectors applied to the evaluation rows, then decoding) and the final assembly. The full rebuild from the raw tables is `run_all.sh` (see `README.md` in the repository).

In [ ]:
# === Cell 1: Setup & Imports ===
import os, sys, json, time, shutil, tarfile, hashlib, subprocess
from pathlib import Path

NOTEBOOK_NAME = 'TPDS_09_submit_exp065_replay_selected.ipynb'
_cell_times = {}
_cell_start = None


def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'▶ {name}')


def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'✓ {name} — {elapsed:.1f}s ({elapsed / 60:.1f}min)')


def run(cmd, cwd=None, env=None):
    """Run a shell command and stream its output line by line; raise if it fails."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'command failed with exit code {rc}: {cmd}')


def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()


cell_start('Cell 1: Setup & Imports')
print('python', sys.version.split()[0], '|', os.cpu_count(), 'CPUs')
cell_end('Cell 1: Setup & Imports')

In [ ]:
# === Cell 2: Config ===
cell_start('Cell 2: Config')
INPUT = Path('/kaggle/input')
COMP = sorted(INPUT.rglob('evaluation_pairs.csv'))[0].parent            # competition data folder
ART = next((d for d in sorted(INPUT.rglob('*')) if d.is_dir() and d.name == 'tpds-9th-place-artifacts'), None)
if ART is None:                                                          # fall back to the folder that holds code.tar
    ART = sorted(INPUT.rglob('code.tar*'))[0].parent
WORK = Path('/tmp/tpds')                 # work tree (checkpoint + code); not saved as notebook output
VENV = Path('/tmp/venv')
OUT = Path('/kaggle/working')
SUBMITTED_A = 'submission_TWFWDp_HB3k12_FINALD2_split_f2p_kdt3_w90.csv'
EXPECTED = {
    'base': ('5_outputs/submissions/submission_FINAL_D2_v040ev_famclf.csv', '92de95cce5ae8d8451f9f77b6f5e1a09'),
    'B': ('5_outputs/submissions/submission_HB3k12_FINALD2_split_f2p_kdt3_w90.csv', 'f17b3b4de833994b8ec994546f3f802b'),
    'A': ('submission.csv', 'c303970ed74faa833cf671ebdfd4efb6'),
}
print('competition data:', COMP)
print('artifacts:       ', ART)
print('work dir:        ', WORK, '| output dir', OUT, '|', os.cpu_count(), 'CPUs')
cell_end('Cell 2: Config')

In [ ]:
# === Cell 3: Code and data ===
cell_start('Cell 3: Code and data')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)


def unpack(name, marker, depth):
    """Extract <name>.tar from the dataset into WORK, or copy the folder Kaggle already extracted it into."""
    tars = [p for p in sorted(ART.rglob(f'{name}.tar')) if p.is_file()]
    if tars:
        with tarfile.open(tars[0]) as t:
            try:
                t.extractall(WORK, filter='data')
            except TypeError:
                t.extractall(WORK)
        return f'extracted {tars[0]}'
    hit = sorted(ART.rglob(marker))[0]
    root = hit.parents[depth]
    shutil.copytree(root, WORK, dirs_exist_ok=True)
    return f'copied {root}'


print(unpack('code', 'replay.sh', 0))
print(unpack('checkpoint', 'cand_pairs_development.parquet', 2))
raw = WORK / '1_data/raw/detect-suspicious-value-transfers-in-poker'
raw.mkdir(parents=True, exist_ok=True)
for f in sorted(COMP.iterdir()):
    (raw / f.name).symlink_to(f)
print('files in the work dir:', sum(1 for p in WORK.rglob('*') if p.is_file()))
cell_end('Cell 3: Code and data')

In [ ]:
# === Cell 4: Python 3.13 environment ===
cell_start('Cell 4: Python 3.13 environment')
run('pip install -q uv')
run(f'uv venv {VENV} --python 3.13 --quiet')
run(f'uv pip install --python {VENV}/bin/python --quiet -r {WORK}/requirements.txt')
PY = f'{VENV}/bin/python'
run(f'{PY} -c "import sys, numpy, polars, pandas, pyarrow, scipy, sklearn, lightgbm, catboost, phevaluator; '
    f'print(sys.version.split()[0], numpy.__version__, polars.__version__, pandas.__version__, pyarrow.__version__, '
    f'scipy.__version__, sklearn.__version__, lightgbm.__version__, catboost.__version__)"')
cell_end('Cell 4: Python 3.13 environment')

In [ ]:
# === Cell 5: Replay ===
cell_start('Cell 5: Replay')
env = dict(os.environ, PY=PY)             # replay.sh sets POLARS_MAX_THREADS=6, as on the reference machine
try:
    run('bash replay.sh', cwd=WORK, env=env)
finally:                                  # keep the step logs as notebook output even if a step fails
    if (WORK / 'logs_replay').exists():
        shutil.copytree(WORK / 'logs_replay', OUT / 'logs_replay', dirs_exist_ok=True)
cell_end('Cell 5: Replay')

In [ ]:
# === Cell 6: Comparison with the submitted files, outputs ===
cell_start('Cell 6: Comparison and outputs')
results = {}
for label, (rel, want) in EXPECTED.items():
    got = md5(WORK / rel)
    results[label] = {'file': rel, 'md5': got, 'identical': got == want}
    print(f"{label}: {'identical to the submitted file' if got == want else 'DIFFERENT'} ({got})")
agreement = None
if not results['A']['identical']:
    submitted_a = sorted(ART.rglob(SUBMITTED_A))[0]
    agreement = subprocess.run([PY, str(WORK / 'tools/compare_submissions.py'), str(submitted_a), str(WORK / 'submission.csv')],
                               capture_output=True, text=True, check=True).stdout
    print(agreement)
for label, (rel, _) in EXPECTED.items():
    shutil.copy(WORK / rel, OUT / Path(rel).name)
shutil.copytree(WORK / 'logs_replay', OUT / 'logs_replay', dirs_exist_ok=True)
json.dump({'md5': results, 'agreement_with_submitted_A': agreement}, open(OUT / 'replay_summary.json', 'w'), indent=1)
cell_end('Cell 6: Comparison and outputs')

In [ ]:
# === Cell 7: Summary ===
cell_start('Summary')
n_same = sum(v['identical'] for v in results.values())
status = 'all three files byte-identical to the submitted ones' if n_same == 3 else f'{n_same}/3 byte-identical (agreement in replay_summary.json)'
print(f"""notebook: {NOTEBOOK_NAME}
DRY_RUN: False
status: {status}
md5_identical: base={results['base']['identical']}, B={results['B']['identical']}, A={results['A']['identical']}
submission: {OUT / 'submission.csv'}
log: {OUT / 'logs_replay'}
summary: {OUT / 'replay_summary.json'}""")
cell_end('Summary')